In [0]:
# %pip install pytest
dbutils.library.restartPython()

In [0]:
import os

candidates = [
    "/Workspace/Users/ibrahimothmanre@gmail.com/ea_data_mgmt",
    "/Workspace/Repos/ibrahimothmanre@gmail.com/ea_data_mgmt",
]

for path in candidates:
    print(path, "→", os.path.exists(path))

print()
print("cwd:", os.getcwd())

In [0]:
import sys
import pytest

BASE = "/Workspace/Repos/ibrahimothmanre@gmail.com/ea_data_mgmt"

if BASE not in sys.path:
    sys.path.insert(0, BASE)

# Workspace paths do not support the operations Python uses to write
# bytecode caches, so disable them.
sys.dont_write_bytecode = True

pytest.main([
    f"{BASE}/tests",
    "-v",
    "-p", "no:cacheprovider",   # stops pytest writing .pytest_cache
])

In [0]:
import sys
print(sys.version)

In [0]:
import os

BASE = "/Workspace/Repos/ibrahimothmanre@gmail.com/ea_data_mgmt"
TESTS = f"{BASE}/tests"

os.makedirs(TESTS, exist_ok=True)

files = {}


files["conftest.py"] = '''"""Shared test fixtures.

These tests cover the pure-Python functions only: no Spark session, no
Delta tables, no notebook. They run with plain pytest.
"""

import sys
from pathlib import Path

import pytest

# Make the package importable without installing it.
REPO_ROOT = Path(__file__).resolve().parent.parent
sys.path.insert(0, str(REPO_ROOT))

from ea_pipeline.config import DatasetContract  # noqa: E402


@pytest.fixture
def strict_contract():
    """A contract that rejects unexpected columns."""
    return DatasetContract(
        required=frozenset({"project_id", "project_name", "project_status"}),
        optional=frozenset({"start_date", "end_date"}),
        reject_unexpected=True,
    )


@pytest.fixture
def lenient_contract():
    """A contract that allows unexpected columns - matches production."""
    return DatasetContract(
        required=frozenset({"project_id", "project_name", "project_status"}),
        optional=frozenset({"start_date", "end_date"}),
        reject_unexpected=False,
    )


@pytest.fixture
def write_csv(tmp_path):
    """
    Write a CSV and return its path.

    tmp_path gives a fresh directory per test, so files never leak
    between tests.
    """
    def _write(content, name="test.csv", encoding="utf-8"):
        path = tmp_path / name
        path.write_text(content, encoding=encoding, newline="")
        return str(path)

    return _write
'''


files["test_normalise_column_name.py"] = '''"""Tests for column name normalisation.

This function is why validation stopped rejecting legitimate Excel
exports: "Project ID" must match a contract expecting "project_id".
"""

import pytest

from ea_pipeline.files import normalise_column_name


@pytest.mark.parametrize(
    "raw,expected",
    [
        ("project_id", "project_id"),
        ("Project ID", "project_id"),
        ("PROJECT ID", "project_id"),
        ("  project_id  ", "project_id"),
        ("project-id", "project_id"),
        ("project.id", "project_id"),
        ("project/id", "project_id"),
        ("Project - ID", "project_id"),
        ("Project   ID", "project_id"),
        ("_project_id_", "project_id"),
    ],
)
def test_normalises_to_expected(raw, expected):
    assert normalise_column_name(raw) == expected


def test_excel_header_matches_contract_name():
    """The specific bug this function was written to fix."""
    assert normalise_column_name("Project ID") == "project_id"


def test_distinct_names_stay_distinct():
    assert normalise_column_name("start_date") != normalise_column_name("end_date")


def test_empty_string():
    assert normalise_column_name("") == ""


def test_only_separators_collapses_to_empty():
    """Guards the blank-column-name check in read_csv_structure."""
    assert normalise_column_name("   ") == ""
    assert normalise_column_name("---") == ""
'''


files["test_read_csv_structure.py"] = '''"""Tests for reading a CSV header and row count.

Every failure here must be a ContractViolation, because those map to
REJECTED rather than FAILED - the file is wrong, and retrying the same
file will not help.
"""

import pytest

from ea_pipeline.errors import ContractViolation
from ea_pipeline.files import read_csv_structure

CLEAN = (
    "project_id,project_name,project_status\\n"
    "P001,Website,Active\\n"
    "P002,Mobile App,Active\\n"
)


def test_reads_header_and_counts_rows(write_csv):
    structure = read_csv_structure(write_csv(CLEAN))

    assert structure.header == ["project_id", "project_name", "project_status"]
    assert structure.row_count == 2      # excludes the header


def test_header_is_normalised(write_csv):
    content = "Project ID,Project Name\\nP001,Website\\n"
    structure = read_csv_structure(write_csv(content))

    assert structure.header == ["project_id", "project_name"]


def test_completely_empty_file(write_csv):
    with pytest.raises(ContractViolation, match="completely empty"):
        read_csv_structure(write_csv(""))


def test_header_but_no_data_rows(write_csv):
    """A common real mistake: exported with a filter still applied."""
    content = "project_id,project_name,project_status\\n"

    with pytest.raises(ContractViolation, match="no data rows"):
        read_csv_structure(write_csv(content))


def test_blank_column_name(write_csv):
    content = "project_id,,project_status\\nP001,x,Active\\n"

    with pytest.raises(ContractViolation, match="blank column name"):
        read_csv_structure(write_csv(content))


def test_semicolon_delimiter_is_detected(write_csv):
    """
    Without this check the error would list every required column as
    missing, which is true but useless to the analyst.
    """
    content = "project_id;project_name;project_status\\nP001;Website;Active\\n"

    with pytest.raises(ContractViolation, match="comma-separated"):
        read_csv_structure(write_csv(content))


def test_tab_delimiter_is_detected(write_csv):
    content = "project_id\\tproject_name\\nP001\\tWebsite\\n"

    with pytest.raises(ContractViolation, match="comma-separated"):
        read_csv_structure(write_csv(content))


def test_non_utf8_file(write_csv):
    """Older Excel exports cp1252, which must not surface as FAILED."""
    content = "project_id,project_name\\nP001,Cafe\\u0301\\n"

    with pytest.raises(ContractViolation, match="not valid UTF-8"):
        read_csv_structure(write_csv(content, encoding="cp1252"))


def test_utf8_bom_is_tolerated(write_csv):
    """Excel CSV UTF-8 adds a BOM; it must not corrupt the first name."""
    content = "project_id,project_name\\nP001,Website\\n"
    path = write_csv(content, encoding="utf-8-sig")

    structure = read_csv_structure(path)

    assert structure.header[0] == "project_id"


def test_quoted_line_break_counts_as_one_row(write_csv):
    """
    A line break inside a quoted value is one row, not two. This count
    must agree with Spark multiLine reader.
    """
    content = (
        "project_id,project_name\\n"
        \'P001,"Website\\nRebuild"\\n\'
        "P002,Mobile App\\n"
    )

    structure = read_csv_structure(write_csv(content))

    assert structure.row_count == 2      # not 3


def test_duplicate_columns_are_preserved(write_csv):
    """
    The header is returned as a list, not a set, so validate_columns can
    still detect the repeat.
    """
    content = "project_id,project_name,project_id\\nP001,Website,P001\\n"

    structure = read_csv_structure(write_csv(content))

    assert structure.header.count("project_id") == 2
'''


files["test_validate_columns.py"] = '''"""Tests for comparing a header against a dataset contract.

validate_columns takes a list of names and a contract and returns a
judgement - no file access, no table lookup - which is what makes these
tests possible without Spark.
"""

import pytest

from ea_pipeline.config import DatasetContract
from ea_pipeline.validate import validate_columns


def test_exact_required_columns_pass(strict_contract):
    result = validate_columns(
        ["project_id", "project_name", "project_status"], strict_contract
    )

    assert result.is_valid
    assert result.missing_columns == []
    assert result.unexpected_columns == []
    assert result.duplicate_columns == []


def test_required_plus_optional_passes(strict_contract):
    result = validate_columns(
        ["project_id", "project_name", "project_status", "start_date"],
        strict_contract,
    )

    assert result.is_valid


def test_column_order_does_not_matter(strict_contract):
    result = validate_columns(
        ["project_status", "project_id", "project_name"], strict_contract
    )

    assert result.is_valid


def test_missing_required_column_fails(strict_contract):
    result = validate_columns(["project_id", "project_name"], strict_contract)

    assert not result.is_valid
    assert result.missing_columns == ["project_status"]


def test_missing_columns_are_sorted(strict_contract):
    result = validate_columns(["project_status"], strict_contract)

    assert result.missing_columns == ["project_id", "project_name"]


def test_duplicate_column_fails(strict_contract):
    result = validate_columns(
        ["project_id", "project_name", "project_status", "project_id"],
        strict_contract,
    )

    assert not result.is_valid
    assert result.duplicate_columns == ["project_id"]


def test_duplicates_detected_even_when_otherwise_valid(lenient_contract):
    """A set would have lost the repeat entirely."""
    result = validate_columns(
        ["project_id", "project_id", "project_name", "project_status"],
        lenient_contract,
    )

    assert not result.is_valid
    assert result.duplicate_columns == ["project_id"]


def test_unexpected_column_rejected_when_strict(strict_contract):
    result = validate_columns(
        ["project_id", "project_name", "project_status", "surprise"],
        strict_contract,
    )

    assert not result.is_valid
    assert result.unexpected_columns == ["surprise"]


def test_unexpected_column_allowed_when_lenient(lenient_contract):
    """
    Source systems add columns routinely; blocking on that would halt
    the pipeline for a harmless change.
    """
    result = validate_columns(
        ["project_id", "project_name", "project_status", "surprise"],
        lenient_contract,
    )

    assert result.is_valid
    assert result.unexpected_columns == ["surprise"]      # still recorded


def test_lenient_contract_still_fails_on_missing(lenient_contract):
    """reject_unexpected must not weaken the required-column check."""
    result = validate_columns(["project_id", "surprise"], lenient_contract)

    assert not result.is_valid
    assert "project_name" in result.missing_columns


def test_actual_columns_preserves_input_order(strict_contract):
    columns = ["project_status", "project_id", "project_name"]
    result = validate_columns(columns, strict_contract)

    assert result.actual_columns == columns


def test_rejection_flag_is_carried_through(lenient_contract):
    """The message builder needs this to word extras as a warning."""
    result = validate_columns(["project_id"], lenient_contract)

    assert result.unexpected_columns_rejected is False


def test_empty_header_reports_all_required_missing(strict_contract):
    result = validate_columns([], strict_contract)

    assert not result.is_valid
    assert len(result.missing_columns) == 3


def test_contract_rejects_overlapping_columns():
    """A column in both required and optional is a config bug."""
    with pytest.raises(ValueError, match="both required and optional"):
        DatasetContract(
            required=frozenset({"project_id"}),
            optional=frozenset({"project_id"}),
        )


def test_contract_rejects_empty_required():
    with pytest.raises(ValueError, match="at least one required"):
        DatasetContract(required=frozenset())
'''


files["test_models.py"] = '''"""Tests for the result dataclasses.

The point is shape consistency: every exit path from a stage must return
the same fields, so callers never branch on which path ran.
"""

from ea_pipeline.models import BronzeOutcome, ColumnCheckResult, ValidationOutcome
from ea_pipeline.states import UploadStatus


def test_rejected_outcome_has_same_fields_as_success():
    """
    The bug this prevents: a caller reading outcome.column_check and
    getting AttributeError only on the rejection path.
    """
    rejected = ValidationOutcome.rejected("abc", "projects", "empty file")

    assert rejected.column_check.missing_columns == []
    assert rejected.column_check.unexpected_columns == []
    assert rejected.column_check.duplicate_columns == []
    assert rejected.source_row_count == 0


def test_rejected_outcome_status():
    outcome = ValidationOutcome.rejected("abc", "projects", "empty file")

    assert outcome.status == UploadStatus.REJECTED
    assert not outcome.is_valid


def test_validated_outcome_is_valid():
    outcome = ValidationOutcome(
        upload_id="abc",
        dataset_name="projects",
        status=UploadStatus.VALIDATED,
        validation_message="ok",
        column_check=ColumnCheckResult(is_valid=True),
        source_row_count=10,
    )

    assert outcome.is_valid


def test_outcomes_are_immutable():
    outcome = ValidationOutcome.rejected("abc", "projects", "empty")

    try:
        outcome.status = UploadStatus.VALIDATED
        raise AssertionError("frozen dataclass should not allow assignment")
    except AttributeError:
        pass


def test_column_check_defaults_are_independent():
    """
    field(default_factory=list) rather than a shared mutable default -
    two instances must not share one list.
    """
    first = ColumnCheckResult(is_valid=True)
    second = ColumnCheckResult(is_valid=True)

    first.missing_columns.append("leaked")

    assert second.missing_columns == []


def test_manifest_update_returns_plain_python():
    """
    models.py must stay Spark-free: the values here are plain Python,
    and manifest.py converts them to column expressions.
    """
    outcome = ValidationOutcome.rejected("abc", "projects", "empty file")
    update = outcome.as_manifest_update()

    assert isinstance(update["status"], str)
    assert isinstance(update["missing_columns"], list)


def test_bronze_rejected_has_zero_counts():
    outcome = BronzeOutcome.rejected(
        "abc", "projects", "ea_dev.bronze.projects", "file changed"
    )

    assert outcome.row_count == 0
    assert outcome.corrupt_row_count == 0
    assert not outcome.succeeded


def test_bronze_success_flag():
    outcome = BronzeOutcome(
        upload_id="abc",
        dataset_name="projects",
        bronze_table="ea_dev.bronze.projects",
        status=UploadStatus.PROCESSED,
        message="ok",
        row_count=10,
    )

    assert outcome.succeeded
'''


for name, content in files.items():
    path = f"{TESTS}/{name}"
    with open(path, "w") as f:
        f.write(content)
    print(f"wrote {name} ({len(content)} chars)")

print()
print("contents:", sorted(os.listdir(TESTS)))

In [0]:
import os, sys, pytest

BASE = "/Workspace/Repos/ibrahimothmanre@gmail.com/ea_data_mgmt"
TESTS = f"{BASE}/tests"

os.environ["PYTHONDONTWRITEBYTECODE"] = "1"
sys.dont_write_bytecode = True

if BASE not in sys.path:
    sys.path.insert(0, BASE)

pytest.main([TESTS, "-v", "-p", "no:cacheprovider"])